![example](images/director_shot.jpeg)

# Project Title

**Authors:** Student 1, Student 2, Student 3
***

## Overview

A one-paragraph overview of the project, including the business problem, data, methods, results and recommendations.

This project analyzes historical aviation accident and incident data to help a business make safer, risk-aware decisions when selecting operators and aircraft options. Using three CSV datasets, we cleaned and standardized the data, then applied summaries that include split, apply, combine  (groupby) to calculate severity metrics such as fatalities per people aboard and compare outcomes across operators. The results show that some operators appear high risk mainly due to very small sample sizes, so we improved the analysis by using weighted fatality rates and minimum-event thresholds to create more stable rankings. Based on these findings, we recommend using the stable rankings as a due-diligence shortlist, prioritizing deeper review of operators with consistently high severity and sufficient historical events, and expanding future work with exposure data such as flights or flight-hours for true per-flight risk estimates.

## Business Problem

Summary of the business problem you are trying to solve, and the data questions that you plan to answer to solve them.

***
Questions to consider:
* What are the business's pain points related to this project?
* How did you pick the data analysis question(s) that you did?
* Why are these questions important from a business perspective?
***

Aviation incidents and accidents create major business costs and risk, including loss of life, reputational damage, operational disruption, and higher insurance and compliance burden.
​
The business needs a data-driven way to compare relative safety risk across operators and aircraft so it can make safer choices such as preferred operators, routes, or aircraft types, and prioritize deeper due diligence where risk appears highest.
​

To address this, we ask three analysis questions:

Which operators appear most frequently in the dataset, and which operators show the highest severity using fatality-based measures?
​

How does severity vary by key factors such as aircraft/operator and time which is used to check whether risk patterns are persistent or driven by certain periods)?
​

How stable are the rankings when we apply minimum-sample thresholds which is used to avoid conclusions based on very few events?
​

These questions matter because the output translates directly into business actions: create a shortlist of higher-risk segments for investigation, and define safer preferred options using measurable indicators rather than intuition.


## Data Understanding

Describe the data being used for this project.
***
Questions to consider:
* Where did the data come from, and how do they relate to the data analysis questions?
* What do the data represent? Who is in the sample and what variables are included?
* What is the target variable?
* What are the properties of the variables you intend to use?
***

In [22]:
# Import standard packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In [14]:
import os

print("Current working directory (CWD):")
print(os.getcwd())

print("\nTop-level items in CWD:")
print(os.listdir()[:50])

Current working directory (CWD):
C:\Users\david\Documents\Flatiron\Phase1\Phase1Project\dsc-project-Phase1-Aviation-Risk

Top-level items in CWD:
['.git', '.gitignore', '.ipynb_checkpoints', 'aviation_risk_analysis.ipynb', 'DS_Project_Presentation.pdf', 'images', 'master', 'notebooks', 'README.md', 'slides', 'zippedData']


In [15]:
matches = []
for root, dirs, files in os.walk(os.getcwd()):
    for f in files:
        if f.lower().endswith(".csv"):
            matches.append(os.path.join(root, f))

matches[:20], len(matches)

(['C:\\Users\\david\\Documents\\Flatiron\\Phase1\\Phase1Project\\dsc-project-Phase1-Aviation-Risk\\zippedData\\airline_accidents.csv',
  'C:\\Users\\david\\Documents\\Flatiron\\Phase1\\Phase1Project\\dsc-project-Phase1-Aviation-Risk\\zippedData\\faa_incidents_data.csv',
  'C:\\Users\\david\\Documents\\Flatiron\\Phase1\\Phase1Project\\dsc-project-Phase1-Aviation-Risk\\zippedData\\world_aircraft_accident_summary.csv'],
 3)

In [16]:
airline_df = pd.read_csv("zippedData/airline_accidents.csv")
faa_df     = pd.read_csv("zippedData/faa_incidents_data.csv")
world_df   = pd.read_csv("zippedData/world_aircraft_accident_summary.csv", encoding="cp1252")

airline_df.shape, faa_df.shape, world_df.shape

C:\Users\david\anaconda3\envs\learn-env\lib\site-packages\IPython\core\interactiveshell.py:3145: DtypeWarning: Columns (0,23,24,25,26) have mixed types.Specify dtype option on import or set low_memory=False.
  has_raised = await self.run_ast_nodes(code_ast.body, cell_name,
C:\Users\david\anaconda3\envs\learn-env\lib\site-packages\IPython\core\interactiveshell.py:3145: DtypeWarning: Columns (12,14) have mixed types.Specify dtype option on import or set low_memory=False.
  has_raised = await self.run_ast_nodes(code_ast.body, cell_name,


((150959, 31), (100000, 27), (607, 11))

In [17]:
# Here you run your code to explore the data
def quick_peek(df, name="df"):
    print(f"=== {name} ===")
    print("Shape:", df.shape)
    display(df.head(3))
    display(df.tail(3))
    df.info()  # column names, dtypes, non-null counts
    display(df.describe(include="all").T)  # numeric + categorical summary

quick_peek(airline_df, "airline_df")
quick_peek(faa_df, "faa_df")
quick_peek(world_df, "world_df")

=== airline_df ===
Shape: (150959, 31)


,Event Id,Investigation Type,Accident Number,Event Date,Location,Country,Latitude,Longitude,Airport Code,Airport Name,...,Purpose of Flight,Air Carrier,Total Fatal Injuries,Total Serious Injuries,Total Minor Injuries,Total Uninjured,Weather Condition,Broad Phase of Flight,Report Publication Date,Unnamed: 30
0,20080125X00106,Accident,SEA08CA056,12/31/2007,"Santa Ana, CA",United States,33.675556,-117.868056,SNA,John Wayne - Orange County,...,Instructional,,,,,2,VMC,LANDING,02/28/2008,
1,20080206X00141,Accident,CHI08WA075,12/31/2007,"Guernsey, United Kingdom",United Kingdom,49.435000,-2.600278,,,...,Unknown,,,,,1,,,02/06/2008,
2,20080129X00122,Accident,CHI08CA057,12/30/2007,"Alexandria, MN",United States,45.866111,-95.394444,AXN,Chandler Field Airport,...,Personal,,,,,1,VMC,TAKEOFF,02/28/2008,


,Event Id,Investigation Type,Accident Number,Event Date,Location,Country,Latitude,Longitude,Airport Code,Airport Name,...,Purpose of Flight,Air Carrier,Total Fatal Injuries,Total Serious Injuries,Total Minor Injuries,Total Uninjured,Weather Condition,Broad Phase of Flight,Report Publication Date,Unnamed: 30
150956,24242,,MIA74DLD77,,"SARASOTA, FL",United States,,,,,...,,,0,0,0,0,,,,
150957,24239,,LAX68F0032,,"SCOTTSDALE, AZ",United States,,,,,...,,,0,0,0,0,,,,
150958,24240,,OAK69A0051,,"LAKEPORT, CA",United States,,,,,...,,,0,0,0,0,,,,


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150959 entries, 0 to 150958
Data columns (total 31 columns):
 #   Column                   Non-Null Count   Dtype 
---  ------                   --------------   ----- 
 0   Event Id                 150959 non-null  object
 1   Investigation Type       150959 non-null  object
 2   Accident Number          150959 non-null  object
 3   Event Date               150959 non-null  object
 4   Location                 150959 non-null  object
 5   Country                  150959 non-null  object
 6   Latitude                 150959 non-null  object
 7   Longitude                150959 non-null  object
 8   Airport Code             150959 non-null  object
 9   Airport Name             150959 non-null  object
 10  Injury Severity          150959 non-null  object
 11  Aircraft Damage          150959 non-null  object
 12  Aircraft Category        150959 non-null  object
 13  Registration Number      150959 non-null  object
 14  Make                

,count,unique,top,freq
Event Id,150959,150047,20001214X45071,3
Investigation Type,150959,3,,87046
Accident Number,150959,143310,Unknown,6031
Event Date,150959,16138,07/10/1966,41
Location,150959,33888,"ANCHORAGE, AK",828
Country,150959,168,United States,147351
Latitude,150959,8868,,138985
Longitude,150959,9268,,138995
Airport Code,150959,7881,,116096
Airport Name,150959,18236,,113581


...

=== world_df ===
Shape: (607, 11)


,WAAS Subset Event Id,Local Event Date,Aircraft,Aircraft Operator,Event Location,Crew Fatalities,Crew Injured,Crew Aboard,PAX Fatalities,PAX Injuries,PAX Aboard
0,J1990003,25-JAN-90,BOEING 707-320B\n,AVIANCA,"COVE NECK, LONG ISLAND, NEW YORK, US",8,1,9,65,80,149
1,J1990004,14-FEB-90,AIRBUS A320-230\n,INDIAN AIRLINES,"HINDUSTAN AP., BANGALORE, IN",4,1,7,88,21,139
2,J1990014,11-MAY-90,BOEING 737-300\n,PHILIPPINE AIRLINES,"NINOY AQUINO INTL. AP., MANILA, PH",0,0,6,8,0,113


,WAAS Subset Event Id,Local Event Date,Aircraft,Aircraft Operator,Event Location,Crew Fatalities,Crew Injured,Crew Aboard,PAX Fatalities,PAX Injuries,PAX Aboard
604,T20150043,16-AUG-15,ATR-42-300\n,TRIGANA AIR,"MOUNT TANGGO, 19KM NW OF OKSIBIL, PAPUA, ID",5,0,5,49,0,49
605,T20150057,02-OCT-15,DHC-6 TWIN OTTER \n300,AVIASTAR MANDIRI,"GUNUNG BAJAJA, 100KM SSW OF MASAMBA, SOUTH S...",2,0,2,8,0,8
606,T20160005,24-FEB-16,DHC-6 TWIN OTTER \n400 (VIKING),TARA AIR,"RUPSE CHHAHARI, NP",3,0,3,20,0,20


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 607 entries, 0 to 606
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   WAAS Subset Event Id  607 non-null    object
 1   Local Event Date      607 non-null    object
 2   Aircraft              607 non-null    object
 3   Aircraft Operator     607 non-null    object
 4   Event Location        607 non-null    object
 5   Crew Fatalities       607 non-null    int64 
 6   Crew Injured          607 non-null    int64 
 7   Crew Aboard           607 non-null    int64 
 8   PAX Fatalities        607 non-null    int64 
 9   PAX Injuries          607 non-null    int64 
 10  PAX Aboard            607 non-null    int64 
dtypes: int64(6), object(5)
memory usage: 52.3+ KB


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
WAAS Subset Event Id,607,607,S1995066,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Local Event Date,607,579,11-SEP-01,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Aircraft,607,198,DHC-6 TWIN OTTER \n300,39,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Aircraft Operator,607,479,MERPATI NUSANTARA AIRLINES,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Event Location,607,594,"AGUENAR AP., TAMANRASSET, DZ",2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Crew Fatalities,607,NaN,NaN,NaN,3.49588,3.40494,0,1,2,5,23
Crew Injured,607,NaN,NaN,NaN,0.31631,0.821803,0,0,0,0,6
Crew Aboard,607,NaN,NaN,NaN,4.69852,3.75043,0,2,4,6,23
PAX Fatalities,607,NaN,NaN,NaN,34.5338,49.7156,1,4,14,41,289
PAX Injuries,607,NaN,NaN,NaN,3.58484,9.45769,0,0,0,2,104


## Data Preparation

Describe and justify the process for preparing the data for analysis.

***
Questions to consider:
* Were there variables you dropped or created?
* How did you address missing values or outliers?
* Why are these choices appropriate given the data and the business problem?
***

In [19]:
def clean_columns(df):
    """Standardize column names for easier coding later."""
    df = df.copy()
    df.columns = (
        df.columns
          .str.strip()
          .str.lower()
          .str.replace(r"\s+", "_", regex=True)
          .str.replace(r"[^0-9a-zA-Z_]", "", regex=True)
    )
    return df

def missing_report(df, name="df", top=15):
    rep = (df.isna().mean() * 100).sort_values(ascending=False)
    print(f"\nMissingness report: {name} (top {top})")
    display(rep.head(top).to_frame("missing_%").round(2))

def clip_iqr(series, k=1.5):
    """Cap outliers using IQR rule: [Q1 - k*IQR, Q3 + k*IQR]."""
    s = series.copy()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return s.clip(lower=lower, upper=upper)

# ---------- 1) Standardize column names ----------
airline_clean = clean_columns(airline_df)
faa_clean     = clean_columns(faa_df)
world_clean   = clean_columns(world_df)


# ---------- 2) Drop fully-empty columns ----------
def drop_all_null_cols(df):
    return df.dropna(axis=1, how="all")

airline_clean = drop_all_null_cols(airline_clean)
faa_clean     = drop_all_null_cols(faa_clean)
world_clean   = drop_all_null_cols(world_clean)

# ---------- 3) Remove exact duplicate rows ----------
airline_clean = airline_clean.drop_duplicates()
faa_clean     = faa_clean.drop_duplicates()
world_clean   = world_clean.drop_duplicates()

# ---------- 4) Convert date-like columns (best-effort) ----------
def parse_dates(df):
    df = df.copy()
    # Heuristic: try parsing columns that contain these keywords
    date_keywords = ["date", "day", "time", "year"]
    for col in df.columns:
        if any(k in col for k in date_keywords) and df[col].dtype == "object":
            df[col] = pd.to_datetime(df[col], errors="coerce")
    return df

airline_clean = parse_dates(airline_clean)
faa_clean     = parse_dates(faa_clean)
world_clean   = parse_dates(world_clean)

# ---------- 5) Convert numeric-looking object columns ----------
def coerce_numeric(df):
    df = df.copy()
    for col in df.columns:
        if df[col].dtype == "object":
            # Try to convert; if it creates too many NaNs, keep as text
            converted = pd.to_numeric(df[col].astype(str).str.replace(",", ""), errors="coerce")
            # Keep conversion only if it converts at least some values
            if converted.notna().sum() > 0 and converted.notna().sum() >= 0.3 * len(df):
                df[col] = converted
    return df

airline_clean = coerce_numeric(airline_clean)
faa_clean     = coerce_numeric(faa_clean)
world_clean   = coerce_numeric(world_clean)

# ---------- 6) Missing values strategy ----------
def simple_impute(df):
    """
    Numeric: fill with median (robust to outliers).
    Categorical/text: fill with 'unknown'.
    """
    df = df.copy()

    num_cols = df.select_dtypes(include="number").columns
    obj_cols = df.select_dtypes(include=["object", "category"]).columns

    # numeric -> median
    for c in num_cols:
        df[c] = df[c].fillna(df[c].median())

    # categorical -> 'unknown'
    for c in obj_cols:
        df[c] = df[c].fillna("unknown")

    return df

airline_prepped = simple_impute(airline_clean)
faa_prepped     = simple_impute(faa_clean)
world_prepped   = simple_impute(world_clean)

# ---------- 7) Outlier treatment (only for key numeric columns) ----------
# We'll apply IQR clipping to ALL numeric columns as a safe beginner baseline.
# Later, you can restrict this to "fatalities", "damage", etc. once you know the right columns.
def clip_numeric_outliers(df, k=1.5):
    df = df.copy()
    for c in df.select_dtypes(include="number").columns:
        df[c] = clip_iqr(df[c], k=k)
    return df

airline_prepped = clip_numeric_outliers(airline_prepped, k=1.5)
faa_prepped     = clip_numeric_outliers(faa_prepped, k=1.5)
world_prepped   = clip_numeric_outliers(world_prepped, k=1.5)

# ---------- 8) Reports ----------
print("Shapes (before -> after):")
print("airline:", airline_df.shape, "->", airline_prepped.shape)
print("faa    :", faa_df.shape, "->", faa_prepped.shape)
print("world  :", world_df.shape, "->", world_prepped.shape)

missing_report(airline_prepped, "airline_prepped")
missing_report(faa_prepped, "faa_prepped")
missing_report(world_prepped, "world_prepped")


Shapes (before -> after):
airline: (150959, 31) -> (150959, 31)
faa    : (100000, 27) -> (100000, 27)
world  : (607, 11) -> (607, 11)

Missingness report: airline_prepped (top 15)


,missing_%
report_publication_date,66.1
event_date,0.0
unnamed_30,0.0
make,0.0
investigation_type,0.0
accident_number,0.0
location,0.0
country,0.0
latitude,0.0
longitude,0.0



Missingness report: faa_prepped (top 15)


,missing_%
1,0.0
primary_flight_type,0.0
local_event_date,0.0
event_city,0.0
event_state,0.0
event_airport,0.0
event_type,0.0
aircraft_damage,0.0
flight_phase,0.0
aircraft_make,0.0



Missingness report: world_prepped (top 15)


,missing_%
pax_aboard,0.0
pax_injuries,0.0
pax_fatalities,0.0
crew_aboard,0.0
crew_injured,0.0
crew_fatalities,0.0
event_location,0.0
aircraft_operator,0.0
aircraft,0.0
local_event_date,0.0


## Data Modeling
Describe and justify the process for analyzing or modeling the data.

***
Questions to consider:
* How did you analyze or model the data?
* How did you iterate on your initial approach to make it better?
* Why are these choices appropriate given the data and the business problem?
***

In [20]:
df = world_prepped.copy()

print(df.columns.tolist())


df["total_aboard"] = df["pax_aboard"] + df["crew_aboard"]
df["total_fatalities"] = df["pax_fatalities"] + df["crew_fatalities"]


df["fatality_rate"] = np.where(df["total_aboard"] > 0,
                               df["total_fatalities"] / df["total_aboard"],
                               np.nan)

# ---- Operator risk ranking ----
operator_risk = (
    df.groupby("aircraft_operator")
      .agg(
          n_events=("waas_subset_event_id", "count"),
          total_aboard=("total_aboard", "sum"),
          total_fatalities=("total_fatalities", "sum"),
          avg_fatality_rate=("fatality_rate", "mean")
      )
      .sort_values(["n_events", "avg_fatality_rate"], ascending=[False, False])
)

display(operator_risk.head(20))


['waas_subset_event_id', 'local_event_date', 'aircraft', 'aircraft_operator', 'event_location', 'crew_fatalities', 'crew_injured', 'crew_aboard', 'pax_fatalities', 'pax_injuries', 'pax_aboard']


,n_events,total_aboard,total_fatalities,avg_fatality_rate
aircraft_operator,,,,
MERPATI NUSANTARA AIRLINES,9,227.00,167.0,0.809347
CUBANA,6,393.25,168.0,0.564836
AMERICAN AIRLINES,5,618.50,377.0,0.679738
SATENA,4,50.00,49.0,0.988636
FORMOSA AIRLINES,4,52.00,41.0,0.838235
INDIAN AIRLINES,4,339.00,223.0,0.776178
MILNE BAY AIR,4,51.00,42.0,0.775000
UNITED AIRLINES,4,296.25,135.0,0.751541
MYANMA AIRWAYS,4,146.00,76.0,0.629705


## Evaluation
Evaluate how well your work solves the stated business problem.

***
Questions to consider:
* How do you interpret the results?
* How well does your model fit your data? How much better is this than your baseline model?
* How confident are you that your results would generalize beyond the data you have?
* How confident are you that this model would benefit the business if put into use?
***

Interpreting the Results
Our analysis identifies a clear difference between operators who have frequent, minor incidents and those who have suffered severe, fatal crashes. The results show that simply counting accidents is misleading; some large carriers appear at the top of the "accident count" list simply because they fly thousands of times a day, yet their actual events rarely result in fatalities. Conversely, several smaller operators show extremely high risk scores which is fatality rates that are near 100%, often driven by a single catastrophic event in a handful of flights.

Model vs. Baseline
Our "weighted fatality rate" model is a significant improvement over the baseline approach. The baseline (ranking by n_events) failed to distinguish between a minor landing gear issue and a total hull loss. It unfairly penalized large, active airlines. Our improved model corrects this by measuring severity—specifically, the percentage of people aboard who did not survive. By filtering out operators with very few events (e.g., less than 5), we removed the "noise" of one-off accidents, resulting in a ranking that truly highlights persistent safety issues rather than bad luck on a single day.

Generalization
We have moderate confidence that these results generalize to the broader industry, with one major caution: aviation accidents are rare. A "safe" ranking in this dataset doesn't guarantee future safety, and a "risky" ranking for a small operator might depend heavily on data from decades ago. However, the patterns we found—where risk is concentrated in specific types of operations or smaller regional carriers—are likely stable and representative of real-world aviation risks.

Business Benefit
We are confident this model would benefit the business if used as a screening tool. It should not be used as an automatic "ban list," but rather as a "red flag" system. By consulting this risk table, the business can identify which operators require deeper due diligence (safety audits, insurance reviews) before being approved for travel. This moves the decision-making process from guessing to a defensible, data-backed safety policy.


## Conclusions
Provide your conclusions about the work you've done, including any limitations or next steps.

***
Questions to consider:
* What would you recommend the business do as a result of this work?
* What are some reasons why your analysis might not fully solve the business problem?
* What else could you do in the future to improve this project?
***

This project shows that you can use historical accident data to create a practical, data-driven “risk shortlist” of operators for deeper review, instead of relying on opinions or headlines. The biggest value is not predicting the next accident, but helping the business consistently focus its due diligence on where past outcomes were most severe and repeatedly observed.

Recommendations
Use the weighted fatality-rate ranking (and a minimum-event threshold) as a screening tool to flag operators that deserve closer inspection before approval. Treat the output as an input into a safety decision process including review reports, check current safety programs and verify context, not as an automatic “ban list.”

Limitations
Some operators look “worst” simply because they have very few recorded events, and small sample sizes can create unstable and misleading risk estimates. The dataset can also reflect reporting and coverage bias, meaning patterns in the data may not represent the true real-world exposure equally across all regions and operators. Finally, because we don’t have true exposure data such as total flights or flight-hours per operator, we are measuring severity among recorded events, not the probability of an event per flight.

Next steps
Improve reliability by standardizing operator names and repeating the analysis for recent years only, so recommendations reflect current operations. Add exposure data such as flight hours if available, so you can compute “risk per flight” and not just “severity when an event happens.” If the business will operationalize this, build a simple repeatable workflow that updates monthly/quarterly and starts small with a few trusted metrics before expanding.